# LSTM Model Multivariate


n this section we implement multivariate forecasting using the LSTM Model (Long Short-Term Memory) with the **TimeSeriesDatasetVectorizedExog** approach.

The LSTM (Long Short-Term Memory) Forecaster is the same univariate model used in the univariate approach, but extended to multivariate forecasting through efficient batching. Instead of processing series individually, **TimeSeriesDatasetVectorizedExog** batches all 1502 series together, allowing the univariate model to train on multiple series simultaneously with exogenous features (GDP, CPI, Interest Rate).

The model architecture  remains unchanged - we simply reshape the data to process all series in parallel, achieving faster training while incorporating exogenous variables. The LSTM's gating mechanisms effectively capture long-term temporal dependencies across all series.

**Layer Breakdown:**

- **LSTM Layers**: 2 stacked LSTM layers with memory cells and three gates (input, forget, output)
- **Hidden Size**: 128 units per layer (default)
- **Dropout**: Applied between LSTM layers (if >1 layer) and before final output
- **Output Layer**: Single fully connected layer producing 1-step forecast
- **Input Features**: Value + GDP + CPI + Interest_Rate + Year + Month + One-Hot Encoding (1502 dims)

In [ ]:
import torch
import torch.nn as nn

## Model

In [ ]:
class LSTMForecaster(nn.Module):
    """
    LSTM model for MULTIVARIATE time series forecasting.
    Architecture: LSTM -> Dropout -> LSTM -> Dropout -> Fully Connected
    Takes multiple input features at each timestep.
    """
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.2):
        """
        Args:
            input_size: Number of input features (Value + year + month + one-hot)
            hidden_size: LSTM hidden dimension
            num_layers: Number of LSTM layers
            dropout: Dropout rate
        """
        super(LSTMForecaster, self).__init__()
        
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.input_size = input_size
        
        # LSTM layers
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )
        
        # Dropout layer
        self.dropout = nn.Dropout(dropout)
        
        # Fully connected output layer
        self.fc = nn.Linear(hidden_size, 1)
    
    def forward(self, x):
        # x shape: (batch_size, seq_length, input_size)
        
        # LSTM forward pass
        lstm_out, (h_n, c_n) = self.lstm(x)
        
        # Take the output from the last time step
        last_output = lstm_out[:, -1, :]  # Shape: (batch_size, hidden_size)
        
        # Apply dropout
        out = self.dropout(last_output)
        
        # Fully connected layer
        out = self.fc(out)  # Shape: (batch_size, 1)
        
        return out

### Model Results without Exogenous Features

### Model Results with Exogenous Features